# Lab type: debug
# Course: ML401 — MLOps & Model Deployment
# Lesson: Model Monitoring and Drift
# Task: The monitoring script below has 3 bugs that would cause incorrect drift detection in production. Identify each bug, explain the incorrect behaviour it produces, and write the corrected code.

In [ ]:
# !pip install evidently pandas numpy scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Generate synthetic dataset representing historical customer data
X, y = make_classification(
    n_samples=10000, n_features=8, n_informative=5,
    random_state=42
)
feature_names = [
    'tenure_days', 'monthly_spend', 'support_tickets_30d',
    'days_since_login', 'contract_months', 'page_views_7d',
    'payment_failures', 'nps_score'
]
df = pd.DataFrame(X, columns=feature_names)
df['churned'] = y
df['event_timestamp'] = pd.date_range('2024-01-01', periods=len(df), freq='1h')

# Split into training, test, and production windows
train_df = df.iloc[:7000].copy()
test_df = df.iloc[7000:8500].copy()

# Production data: 3 months later — some features have drifted
production_df = df.iloc[8500:].copy()
# Simulate drift: tenure_days and monthly_spend have shifted
production_df['tenure_days'] = production_df['tenure_days'] + np.random.normal(0.8, 0.1, len(production_df))
production_df['monthly_spend'] = production_df['monthly_spend'] * 1.3  # 30% increase
production_df['event_timestamp'] = pd.date_range('2024-04-01', periods=len(production_df), freq='1h')

print(f'Training set: {len(train_df)} rows')
print(f'Test set: {len(test_df)} rows')
print(f'Production sample: {len(production_df)} rows')

## The broken monitoring script

The following script is used to generate a daily drift report.
There are **3 bugs**. Identify each one before running the corrected version.

In [ ]:
# === BUGGY MONITORING SCRIPT — find the 3 bugs ===

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    HAS_EVIDENTLY = True
except ImportError:
    HAS_EVIDENTLY = False
    print('Evidently not installed — showing mock analysis only')

# Bug 1: Wrong reference dataset
reference_data = test_df[feature_names]      # using test set as reference
current_data = production_df[feature_names]

# Bug 2: Threshold set to zero
if HAS_EVIDENTLY:
    report = Report(metrics=[
        DataDriftPreset(drift_share_threshold=0.0)  # fires on any drift
    ])
    report.run(
        reference_data=reference_data,
        current_data=current_data
    )
    result = report.as_dict()
    print('Drift detected:', result['metrics'][0]['result']['dataset_drift'])

# Bug 3: Missing timestamp column used for time-windowed monitoring
def compute_rolling_drift_stats(df, window_days=7):
    """Compute feature statistics over a rolling window."""
    # This will fail because event_timestamp was excluded from feature_names
    # and production_df has no timestamp here
    recent = df[df['event_timestamp'] > pd.Timestamp.now() - pd.Timedelta(days=window_days)]
    return recent[feature_names].describe()

# Attempt to use the function
try:
    stats = compute_rolling_drift_stats(production_df)
    print(stats)
except KeyError as e:
    print(f'Error: {e} — timestamp column not available for windowed analysis')

## Your analysis

**Bug 1** (reference dataset):
- What is wrong:
- Incorrect behaviour it produces:
- Correct approach:

**Bug 2** (drift threshold):
- What is wrong:
- Incorrect behaviour it produces in production:
- What threshold would you use instead, and how would you choose it:

**Bug 3** (timestamp handling):
- What is wrong:
- Incorrect behaviour it produces:
- Correct approach:

## Answers

**Bug 1:** `test_df` is used as the reference dataset instead of `train_df`. Drift monitoring measures how much the current production data has shifted from the model's training distribution. The test set is a smaller sample from the same distribution — but it is noisier (1500 rows vs 7000). Using a noisy reference produces false drift alerts on features with high natural variance. The correct reference is the full training dataset.

**Bug 2:** `drift_share_threshold=0.0` means 'flag drift if any share of features shows any statistical deviation'. Natural statistical variation between samples will always produce some test statistic deviation. This threshold will fire on every monitoring run. The operational consequence is alert fatigue — constant alerts that are always false positives, causing the team to ignore them. A typical starting point is `drift_share_threshold=0.5` (flag if more than 50% of features show significant drift), then tune based on observed natural variation in your specific data.

**Bug 3:** The `compute_rolling_drift_stats` function filters on `event_timestamp`, but `production_df` passed to it does not include `event_timestamp` in the subset passed to the function. The function will raise a KeyError when trying to filter. The fix is to include the timestamp column in the dataframe passed to the windowing function, or to pass it as a separate parameter.

In [ ]:
# === CORRECTED MONITORING SCRIPT ===

# Fix 1: Use training data as reference
reference_data_correct = train_df[feature_names]  # training distribution
current_data_correct = production_df[feature_names]

if HAS_EVIDENTLY:
    # Fix 2: Set a meaningful drift threshold
    report_correct = Report(metrics=[
        DataDriftPreset(drift_share_threshold=0.5)  # flag if >50% of features drift
    ])
    report_correct.run(
        reference_data=reference_data_correct,
        current_data=current_data_correct
    )
    result_correct = report_correct.as_dict()
    print('Drift detected (corrected):', result_correct['metrics'][0]['result']['dataset_drift'])
    
    # Show which features drifted
    for feature_metric in result_correct['metrics'][0]['result']['drift_by_columns'].items():
        feat, data = feature_metric
        if data.get('drift_detected'):
            print(f'  Drifted: {feat} (score: {data.get("drift_score", "n/a"):.4f})')
else:
    # Manual drift check without Evidently
    print('Manual drift check (PSI approximation):')
    for feat in feature_names:
        ref_mean = reference_data_correct[feat].mean()
        cur_mean = current_data_correct[feat].mean()
        pct_change = abs(cur_mean - ref_mean) / (abs(ref_mean) + 1e-9)
        status = 'DRIFT' if pct_change > 0.2 else 'ok'
        print(f'  {feat}: ref_mean={ref_mean:.3f}, cur_mean={cur_mean:.3f}, change={pct_change:.1%} [{status}]')

# Fix 3: Include timestamp in windowed analysis
def compute_rolling_drift_stats_correct(df, window_days=7, timestamp_col='event_timestamp'):
    """Compute feature statistics over a rolling window."""
    if timestamp_col not in df.columns:
        raise ValueError(f'Timestamp column {timestamp_col!r} not found in dataframe')
    recent = df[df[timestamp_col] > pd.Timestamp.now() - pd.Timedelta(days=window_days)]
    available_features = [f for f in feature_names if f in df.columns]
    return recent[available_features].describe()

# Use production_df which has event_timestamp
stats = compute_rolling_drift_stats_correct(
    production_df,
    window_days=30,  # use 30 days to capture some production data
)
print('\nProduction feature statistics (recent window):')
print(stats.loc[['mean', 'std']].round(3))

## Extension: compare reference vs production distributions

The code below shows the actual distribution shift for the drifted features.
After running it, answer: given that `monthly_spend` increased by 30%, what action would you recommend and why?

In [ ]:
import matplotlib
matplotlib.use('Agg')  # for Colab compatibility
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, feat in zip(axes, ['tenure_days', 'monthly_spend']):
    ax.hist(train_df[feat], bins=40, alpha=0.6, label='Training', color='steelblue')
    ax.hist(production_df[feat], bins=40, alpha=0.6, label='Production', color='tomato')
    ax.set_title(f'{feat} distribution')
    ax.legend()

plt.tight_layout()
plt.savefig('drift_comparison.png', dpi=100)
plt.show()
print('Distribution comparison saved.')

# Summary statistics comparison
print('\nFeature comparison (training vs production):')
for feat in ['tenure_days', 'monthly_spend']:
    train_mean = train_df[feat].mean()
    prod_mean = production_df[feat].mean()
    print(f'{feat}: training mean={train_mean:.3f}, production mean={prod_mean:.3f}, change={((prod_mean-train_mean)/abs(train_mean)*100):+.1f}%')

## Reflection: recommended action

Given that `monthly_spend` has increased by 30% in the production data compared to the training distribution:

1. Before deciding to retrain, what would you check in the upstream data pipeline to rule out a data bug?

2. If the upstream pipeline looks correct and the increase reflects a genuine pricing change, does this mean the model's churn predictions are wrong? How would you evaluate this?

3. If you decide to retrain, what data would you use as the training set? Would you use the full historical data, only recent data, or a weighted combination? Explain your reasoning.